# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn --quiet

# 1. Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- Special tokens ----
    prediction_start_token: str = "<|prediction|>"
    prediction_end_token: str = "<|/prediction|>"

    # ---- Paths ----
    adapter_path: str = "./outputs/final_adapter"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

# 3. Define Special Tokens

In [ ]:
SPECIAL_TOKENS = [
    # Input structure
    "<|system|>", "<|/system|>",
    "<|conversation|>", "<|/conversation|>",
    "<|agent|>", "<|customer|>",
    # Output structure
    "<|prediction|>", "<|/prediction|>",
]

# 4. Load Eval Data

In [ ]:
import pandas as pd

eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

print(f"Eval: {len(eval_data)} samples")
eval_data.head()

# 5. Load Model + Tokenizer with LoRA Adapter

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer from adapter ----
tokenizer = AutoTokenizer.from_pretrained(cfg.adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocab size: {len(tokenizer)}")

# ---- Load base model ----
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

base_model.resize_token_embeddings(len(tokenizer))

# ---- Load LoRA adapter ----
model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
model.eval()

print(f"Model loaded with LoRA adapter from {cfg.adapter_path}")

# 6. Inference Functions

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert sales call analyst specializing in financial services. "
    "You will be given a full transcript of a customer-agent conversation. Your task is to:\n"
    "Predict whether the customer will convert (1) or not (0) based the conversation."
)


def build_inference_prompt(system_prompt: str, transcript: str) -> str:
    return (
        f"<|system|>{system_prompt}<|/system|>\n"
        f"<|conversation|>\n{transcript}\n<|/conversation|>\n"
        f"<|prediction|>"
    )


def predict(model, tokenizer, transcript: str, max_new_tokens=200):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return {
        "generated_output": new_tokens,
        "full_text": tokenizer.decode(outputs[0], skip_special_tokens=False),
    }


def predict_with_logit_probability(model, tokenizer, transcript: str):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]

    token_0 = tokenizer.encode("0", add_special_tokens=False)[0]
    token_1 = tokenizer.encode("1", add_special_tokens=False)[0]

    binary_logits = logits[:, [token_0, token_1]]
    probs = F.softmax(binary_logits, dim=-1)
    conversion_prob = probs[0, 1].item()

    return {
        "conversion_probability": conversion_prob,
        "predicted_class": 1 if conversion_prob > 0.5 else 0,
        "raw_logits": {"token_0": logits[0, token_0].item(), "token_1": logits[0, token_1].item()},
    }

# 7. Test Inference on One Example

In [ ]:
test_row = eval_data.iloc[0]
test_transcript = test_row["full_text"]
test_label = test_row["outcome"]

print("=" * 60)
print("TEST INFERENCE: Full Generation")
print("=" * 60)
result = predict(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Generated output:\n{result['generated_output']}")

print("\n" + "=" * 60)
print("TEST INFERENCE: Logit-Based Probability")
print("=" * 60)
prob_result = predict_with_logit_probability(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Conversion probability: {prob_result['conversion_probability']:.4f}")
print(f"Predicted class: {prob_result['predicted_class']}")
print(f"Raw logits: {prob_result['raw_logits']}")

# 8. Evaluate on Full Eval Set

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

eval_results = []

print("Running evaluation...")
for i in range(len(eval_data)):
    row = eval_data.iloc[i]
    prob_result = predict_with_logit_probability(
        model, tokenizer, row["full_text"]
    )
    eval_results.append({
        "ground_truth": row["outcome"],
        "predicted_class": prob_result["predicted_class"],
        "probability": prob_result["conversion_probability"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(eval_data)} samples")

# ---- Compute metrics ----
correct = sum(1 for r in eval_results if r["ground_truth"] == r["predicted_class"])
total = len(eval_results)
accuracy = correct / total

y_true = [r["ground_truth"] for r in eval_results]
y_pred = [r["predicted_class"] for r in eval_results]
y_prob = [r["probability"] for r in eval_results]

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"\n{classification_report(y_true, y_pred, target_names=['No Convert (0)', 'Convert (1)'])}")

# 9. Cross-Entropy with Ground Truth (Dirac Delta)

For a Dirac delta ground-truth distribution where $q(y=k) = 1$ for the true class and $0$ otherwise, the cross-entropy simplifies to:

$$H(q, p) = -\log p(y = y_{\text{true}})$$

where $p(y=1)$ is the model's predicted conversion probability. Per-sample cross-entropy is computed and then averaged across the eval set.

In [ ]:
import numpy as np

y_true_arr = np.array(y_true)
y_prob_arr = np.array(y_prob)

# Clip probabilities to avoid log(0)
eps = 1e-7
y_prob_clipped = np.clip(y_prob_arr, eps, 1.0 - eps)

# Per-sample cross-entropy: -log p(y_true)
#   If y_true == 1: CE = -log(p)
#   If y_true == 0: CE = -log(1 - p)
per_sample_ce = -(y_true_arr * np.log(y_prob_clipped) + (1 - y_true_arr) * np.log(1 - y_prob_clipped))

mean_ce = per_sample_ce.mean()

print(f"{'='*60}")
print(f"CROSS-ENTROPY (Dirac Delta Ground Truth)")
print(f"{'='*60}")
print(f"Mean cross-entropy: {mean_ce:.4f}")
print(f"Min  cross-entropy: {per_sample_ce.min():.4f}")
print(f"Max  cross-entropy: {per_sample_ce.max():.4f}")
print(f"Std  cross-entropy: {per_sample_ce.std():.4f}")

# ---- Per-class breakdown ----
ce_class_0 = per_sample_ce[y_true_arr == 0]
ce_class_1 = per_sample_ce[y_true_arr == 1]

print(f"\nPer-class mean cross-entropy:")
print(f"  No Convert (0): {ce_class_0.mean():.4f} (n={len(ce_class_0)})")
print(f"  Convert    (1): {ce_class_1.mean():.4f} (n={len(ce_class_1)})")